In [78]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

In [79]:
df = pd.read_csv('customer_data.csv')

In [80]:
print("Dataset Shape:", df.shape)

Dataset Shape: (580, 8)


In [81]:
print("\nFirst 3 rows:")
print(df.head(3))


First 3 rows:
   CustomerID  Age  Income  SpendingScore  Gender Country Membership Purchase
0           1   25   30000             45    Male     USA       Gold      Yes
1           2   32   45000             60  Female      UK     Silver      Yes
2           3   41   60000             75    Male  Canada       Gold      Yes


In [82]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
CustomerID       0
Age              0
Income           0
SpendingScore    0
Gender           0
Country          0
Membership       0
Purchase         0
dtype: int64


In [83]:
X = df.drop('Purchase', axis=1)
y = df['Purchase']

y = y.map({'Yes': 1, 'No': 0})

In [84]:
num_features = ['Age', 'Income', 'SpendingScore']
cat_features = ['Gender', 'Country', 'Membership']

In [85]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
num_pipeline

,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [86]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
cat_pipeline

,steps,"[('imputer', ...), ('onehot', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,categories,'auto'


In [87]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [88]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [89]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

### Grid Search Parameters

In [90]:
param_grids = {

    "Logistic Regression": {

        "classifier__C": [0.01,0.1,1,10],

        "classifier__solver": ["liblinear"]

    },

    "Decision Tree": {

        "classifier__criterion":["gini","entropy"],

        "classifier__max_depth":[3,5,10,None],

        "classifier__min_samples_split":[2,5,10]

    },

    "Random Forest": {

        "classifier__n_estimators":[100,200],

        "classifier__max_depth":[5,10,None],

        "classifier__min_samples_split":[2,5],

        "classifier__criterion":["gini","entropy"]

    }

}

### Training Loop

In [91]:
best_model = None

best_accuracy = 0

best_model_name = ""

results = []

print("="*60)
print("MODEL TRAINING")
print("="*60)

for name, model in models.items():

    print(f"\nTraining {name}...\n")

    pipeline = Pipeline([

        ("preprocessor", preprocessor),

        ("classifier", model)

    ])

    grid_search = GridSearchCV(

        estimator=pipeline,

        param_grid=param_grids[name],

        cv=5,

        scoring="accuracy",

        n_jobs=-1

    )

    grid_search.fit(X_train, y_train)

    y_pred = grid_search.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results.append({

        "Model":name,

        "Accuracy":accuracy

    })

    print("Best Parameters")

    print(grid_search.best_params_)

    print()

    print(f"Test Accuracy : {accuracy*100:.2f}%")

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        best_model = grid_search.best_estimator_

        best_model_name = name

MODEL TRAINING

Training Logistic Regression...

Best Parameters
{'classifier__C': 0.01, 'classifier__solver': 'liblinear'}

Test Accuracy : 92.24%

Training Decision Tree...

Best Parameters
{'classifier__criterion': 'gini', 'classifier__max_depth': 3, 'classifier__min_samples_split': 2}

Test Accuracy : 93.10%

Training Random Forest...

Best Parameters
{'classifier__criterion': 'gini', 'classifier__max_depth': 5, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}

Test Accuracy : 92.24%


### Best Model

In [92]:
print("\n"+"="*60)

print("BEST MODEL")

print("="*60)

print(f"Model : {best_model_name}")

print(f"Accuracy : {best_accuracy*100:.2f}%")


BEST MODEL
Model : Decision Tree
Accuracy : 93.10%


### Retrain Best Model

In [93]:
best_model.fit(X, y)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [94]:
joblib.dump(

    best_model,

    "customer_purchase_model.joblib"

)

print("\nModel saved successfully.")


Model saved successfully.


In [95]:
report = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(report)

Confusion Matrix:
[[18  6]
 [ 3 89]]


In [96]:
loaded_model = joblib.load(
    "customer_purchase_model.joblib"
)

In [97]:
new_customers = pd.read_csv('new_customers.csv')
new_predictions = loaded_model.predict(new_customers)

new_customers['Purchase_Prediction'] = ['Yes' if p == 1 else 'No' for p in new_predictions]
new_customers.to_string(index=False)
new_customers

,Age,Income,SpendingScore,Gender,Country,Membership,Purchase_Prediction
0,31,42000,55,Male,USA,Gold,Yes
1,29,35000,42,Female,UK,Silver,No
2,44,68000,82,Male,Canada,Platinum,Yes
3,26,29000,36,Female,India,Bronze,No
4,38,56000,64,Male,USA,Silver,Yes
5,33,45000,58,Female,UK,Gold,Yes
6,41,63000,74,Male,Canada,Platinum,Yes
7,24,26000,28,Female,India,Bronze,No
8,47,78000,92,Male,USA,Gold,Yes
9,35,52000,62,Female,UK,Silver,Yes


In [98]:
purchase_summary = new_customers['Purchase_Prediction'].value_counts()
print(f"\nPrediction Summary:")
print(f"Will Purchase:  {purchase_summary.get('Yes', 0)} customers")
print(f"Will Not Purchase: {purchase_summary.get('No', 0)} customers")


Prediction Summary:
Will Purchase:  13 customers
Will Not Purchase: 7 customers


In [99]:
print("\nBUSINESS INSIGHTS")
print("-"*40)
print("Key Factors Influencing Purchase Decisions:")
print("   - Membership type (Gold/Platinum members show higher conversion)")
print("   - Spending Score (higher spending score indicates higher purchase probability)")
print("   - Income level (higher income customers more likely to purchase)")
print("   - Age group (30-45 age range shows best conversion rates)")


BUSINESS INSIGHTS
----------------------------------------
Key Factors Influencing Purchase Decisions:
   - Membership type (Gold/Platinum members show higher conversion)
   - Spending Score (higher spending score indicates higher purchase probability)
   - Income level (higher income customers more likely to purchase)
   - Age group (30-45 age range shows best conversion rates)
